# Étape 4 : contrôles des règles de gestion et rapprochements entre sources

Ce notebook applique aux données chargées les règles de gestion données par le métier dans le formulaire des règles de gestion, puis enregistrées dans `config/gestion.yaml`. Chaque règle est une instance d'un type générique du catalogue, avec ses paramètres (table, colonnes). Les dates, heures et nombres sont lus selon leur nature dans le dictionnaire enregistré (`data/config/dictionnaire.yaml`) : une valeur vide ou illisible rend la ligne non évaluable, sans nouvel écart.

Un écart est une anomalie, reste à arbitrer par le métier (avec, quand c'est possible, une correction proposée), ou est une information sur une pratique normale. Rien n'est corrigé. Les valeurs des colonnes sensibles ne sont jamais affichées.

Avant les contrôles, trois cellules font circuler ce formulaire (phase 6 du mode opératoire). `# EXPORT FORMULAIRE` l'écrit dans `data/formulaires/` s'il n'y est pas déjà, avec les règles en vigueur et des propositions tirées des données, et le métier le remplit dans Excel. `# IMPORT FORMULAIRE` puis `# ECRITURE FORMULAIRE` réécrivent alors `config/gestion.yaml`, après une copie datée de l'ancien fichier, et archivent le formulaire appliqué dans `data/formulaires/appliques/`. Un formulaire pas encore rempli ne change rien, si bien que le notebook peut toujours être exécuté en entier.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord les notebooks `03_dictionnaire` et `04_controles_format_doublon`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 03_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# EXPORT FORMULAIRE

from openpyxl import load_workbook

from frigeo.confidentialite import charger_sensibilite
from frigeo.doublons import charger_declarations_doublons
from frigeo.formulaire_gestion import (
    FEUILLE_MODE_EMPLOI,
    chemin_formulaire_gestion,
    empreinte_gestion,
    exporter_formulaire_gestion,
)
from frigeo.gestion import lire_fichier_gestion

# Le formulaire part des règles en vigueur dans config/gestion.yaml et des données
# chargées. Les clés et colonnes uniques de config/controles.yaml servent de cibles aux
# références proposées, et une colonne sensible ne sert jamais de colonne selon.
# Un formulaire déjà exporté n'est pas remplacé, il attend son import.
chemin_form = chemin_formulaire_gestion(PERIODE_FIN)
if chemin_form.exists():
    print("Formulaire déjà exporté, en attente d'import,", chemin_form)
else:
    exporter_formulaire_gestion(
        tables, dictionnaire, lire_fichier_gestion(), charger_declarations_doublons(),
        charger_sensibilite(), PERIODE_FIN, empreinte=empreinte_gestion(),
    )
    print("Formulaire exporté,", chemin_form)

# Ce qui reste à traiter, tel que le mode d'emploi du formulaire l'affiche.
classeur = load_workbook(chemin_form, read_only=True)
try:
    for (texte,) in classeur[FEUILLE_MODE_EMPLOI].iter_rows(max_col=1, values_only=True):
        if texte and texte.startswith("Feuille") and "observé" in texte:
            print(texte)
finally:
    classeur.close()

# Un classeur d'un autre mois, ou d'un autre formulaire, resté dans le dossier n'a pas
# été importé.
autres = sorted(
    p.name for p in chemin_form.parent.glob("*.xlsx")
    if p != chemin_form and not p.name.startswith("~$")
)
print(len(autres), "autres formulaires en attente", ", ".join(autres))

In [ ]:
# IMPORT FORMULAIRE

from frigeo.formulaire_gestion import (
    chemin_formulaire_gestion,
    empreinte_gestion,
    importer_formulaire_gestion,
)
from frigeo.gestion import lire_fichier_gestion

# Enregistrer et fermer le formulaire dans Excel avant d'exécuter cette cellule.
# Règles en vigueur avant l'import, que la cellule suivante compare au formulaire.
contenu_gestion = lire_fichier_gestion()
empreinte = empreinte_gestion()
chemin_form = chemin_formulaire_gestion(PERIODE_FIN)
formulaire = None
if not chemin_form.exists():
    print("Aucun formulaire à importer,", chemin_form)
else:
    # Un formulaire mal rempli arrête ici, avec la liste des lignes à corriger.
    formulaire = importer_formulaire_gestion(
        chemin_form, dictionnaire, contenu_gestion, empreinte=empreinte
    )
    print(len(formulaire.changements), "changements apportés par le formulaire,",
          len(formulaire.avertissements), "avertissements")
    for avertissement in formulaire.avertissements:
        print(" ", avertissement)
    display(formulaire.changements)

In [ ]:
# ECRITURE FORMULAIRE

from frigeo.formulaire_gestion import appliquer_formulaire_gestion
from frigeo.gestion import chemin_regles_gestion

if formulaire is None:
    print("Aucun formulaire importé, config/gestion.yaml non réécrit.")
else:
    # Le formulaire est archivé dans data/formulaires/appliques, puis le fichier est
    # réécrit après une copie datée dans data/config/sauvegardes.
    application = appliquer_formulaire_gestion(
        formulaire, chemin_form, contenu_gestion, empreinte=empreinte
    )
    if application.archive is None:
        print("Aucune règle changée, config/gestion.yaml non réécrit, formulaire "
              "laissé en place.")
    else:
        print("Réécrit, à commiter,", chemin_regles_gestion())
        print("Copie de l'ancien fichier", application.sauvegarde)
        print("Formulaire archivé dans", application.archive)
    # Une seconde exécution de la cellule ne doit pas appliquer le formulaire deux fois.
    formulaire = None

In [ ]:
# TYPES DE REGLES

from frigeo.gestion import catalogue_types

# Types disponibles pour écrire config/gestion.yaml, avec leurs paramètres.
display(catalogue_types())

In [ ]:
%%time
# REGLES DE GESTION

from frigeo.controles import ISSUE_A_ARBITRER, ISSUE_ANOMALIE
from frigeo.doublons import charger_declarations_doublons
from frigeo.gestion import (
    ISSUE_INFORMATION,
    charger_regles_gestion,
    chemin_regles_gestion,
    controler_gestion,
)

# Règles données par le métier dans config/gestion.yaml (aucune si le fichier est absent).
regles = charger_regles_gestion()
print(len(regles), "règles de gestion lues dans", chemin_regles_gestion())
# La clé de chaque table, affichée avec les écarts, vient des déclarations de doublons.
declarations = charger_declarations_doublons()

# La date de coupure (fin de la période extraite) sert aux délais non échus.
ecarts_gestion, bilan_gestion = controler_gestion(
    tables, dictionnaire, regles, declarations, date_coupure=perimetre.date_coupure
)
print(len(ecarts_gestion), "écarts, dont",
      int((ecarts_gestion["issue"] == ISSUE_ANOMALIE).sum()), "anomalies,",
      int((ecarts_gestion["issue"] == ISSUE_A_ARBITRER).sum()), "à arbitrer et",
      int((ecarts_gestion["issue"] == ISSUE_INFORMATION).sum()), "informations")
print(int(ecarts_gestion["proposition"].notna().sum()), "corrections proposées")

# Avertissements sur la configuration (par exemple une année sans jours fériés déclarés) :
# à lire avant le bilan, ils peuvent rendre des lignes non évaluables.
avertis = bilan_gestion[bilan_gestion["avertissements"].notna()]
print(len(avertis), "règles avec un avertissement")
for regle in avertis.itertuples():
    print(f"  {regle.ref} ({regle.libelle}) : {regle.avertissements}")
display(bilan_gestion.drop(columns=["libelle", "avertissements"]))

In [ ]:
# DETAIL PAR CONSTAT

display(
    ecarts_gestion.groupby(["ref", "table", "constat", "issue"])
    .agg(lignes=("constat", "size"), propositions=("proposition", "count"))
)